# RNNs & LSTMs: Sequential Data Reference

Reach for this when you need: 
- Reference for processing sequences (text, time-series).
- To understand how to handle variable-length sequences with `pack_padded_sequence`.
- Implementation for many-to-one or many-to-many sequence modeling.

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Core Recurrent Layers

| Layer | Description | Industry Usage |
| :--- | :--- | :--- |
| `nn.RNN` | Vanilla recurrence | Rarely used due to vanishing gradients |
| `nn.LSTM` | Long Short-Term Memory | SOTA for smaller sequential tasks (pre-Transformers) |
| `nn.GRU` | Gated Recurrent Unit | Faster, simpler alternative to LSTM |
| `nn.utils.rnn.pack_padded_sequence` | Speeds up processing of padded batches | Critical for efficiency with variable length samples |

In [ ]:
# nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True)

# Input: (Batch, Sequence, Features)
x = torch.randn(5, 100, 10) # 5 samples, 100 timesteps, 10 features

# Output: (Output, (hidden_state, cell_state))
out, (hn, cn) = lstm(x)

print(out.shape) # [5, 100, 20] - Sequence of all hidden states
print(hn.shape)  # [2, 5, 20] - Last hidden state (2 layers)

## 2. Bidirectional RNNs

Allows the model to see context 'from the future' (useful for non-causal NLP like NER/Tagging).

✅ **Use when**: Analyzing static sequences (e.g. classification of full sentences).
❌ **Don't use when**: Generative tasks where future context is unknown (e.g. LLM next-word prediction).

In [ ]:
bi_lstm = nn.LSTM(10, 20, bidirectional=True, batch_first=True)
out, (hn, cn) = bi_lstm(x)

# Output size is doubled to 40 because it concatenates forward and backward states
print(out.shape) # [5, 100, 40]

### Common Pitfalls
- **batch_first=False**: PyTorch RNNs default to `(Seq, Batch, Feature)`. ALWAYS set `batch_first=True` if your DataLoader provides `(Batch, Seq, Feature)`.
- **Hidden state initialization**: You don't HAVE to initialize (it defaults to zeros), but if you are doing stateful training over multiple batches, you must pass the previous `(hn, cn)` to the next call and `.detach()` them.
- **Output Shape**: Remember that `out` contains ALL timesteps, while `hn` contains ONLY the last timestep for each layer.

### Key Takeaways
- LSTMs solve the vanishing gradient problem via the 'Cell State' (gate-controlled memory line).
- Use `num_layers` to stack units for deeper hierarchical feature extraction.
- For many-to-one tasks (classification), use the LAST hidden state: `out[:, -1, :]` or `hn[-1]` (depending on layer count).